# 02 — Cleaning and filtering

Apply the same detection and signal-to-noise requirements to training and test
light curves. Retain detected observations with absolute SNR above three and
objects with at least five retained observations. Synchronize object metadata.
Keep negative differential-flux observations when they pass these requirements.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))  

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src.data.cleaning import cleaning_pipeline, synchronize_metadata


## 1. Load the data


In [ ]:
df = pd.read_csv(config.TRAIN_CSV)
metadata_table = pd.read_csv(config.METADATA_CSV)
print(f'crudo: {len(df):,} obs · {df.object_id.nunique():,} objects')


## 2. Apply the cleaning pipeline


In [ ]:
df_clean = cleaning_pipeline(df, snr_min=config.SNR_MIN, min_obs=config.MIN_OBS)


### Sequential impact of filtering


In [ ]:
n_obs_raw, n_obj_raw = len(df), df.object_id.nunique()
n_obs_cln, n_obj_cln = len(df_clean), df_clean.object_id.nunique()
print(f"Observaciones: {n_obs_raw:,} -> {n_obs_cln:,}  "
      f"({100*(1-n_obs_cln/n_obs_raw):.0f}% descartadas by SNR < {config.SNR_MIN})")
print(f"Objetos:       {n_obj_raw:,} -> {n_obj_cln:,}  "
      f"({n_obj_raw-n_obj_cln:,} eliminados by quedar with < {config.MIN_OBS} obs)")
print(f"Flux negativo conservado: {100*(df_clean.flux<0).mean():.0f}% of the observations limpias "
      f"(the limpieza not descarta the señal diferencial negativa)")


## 3. Synchronize metadata


In [ ]:
meta_clean = synchronize_metadata(metadata_table, df_clean)
assert set(df_clean.object_id) == set(meta_clean.object_id)
print(f'objects finales: {meta_clean.object_id.nunique():,}')


## 4. Save the cleaned tables


In [ ]:
df_clean.to_csv(config.TRAIN_CLEAN_CSV, index=False)
meta_clean.to_csv(config.METADATA_CLEAN_CSV, index=False)
print('Saved in data/processed/')
